# Lab 2: Text Pre-processing and Regular Expressions

Complete the practical exercise at the end of the lab: find how many hashtags occur in the Apple tweets dataset and display the top 10 hashtags. This notebook follows the same task headings, short explanations, and focused code cells as Lab 1.

## Task 1: Download and Open the Dataset

- **Dataset:** Apple Twitter Sentiment Texts
- **Source:** [Kaggle dataset](https://www.kaggle.com/datasets/seriousran/appletwittersentimenttexts)
- **Purpose:** Use regular expressions to extract hashtags from tweet text.

### Install Libraries
`kagglehub` downloads the files. `pandas` reads the CSV and presents the results. Regular expressions and frequency counting use Python's built-in libraries.

In [ ]:
%pip install -q kagglehub pandas

### Import Libraries

In [ ]:
from collections import Counter
from pathlib import Path
import re

import kagglehub
import pandas as pd

### Download the Dataset
If you download the dataset manually from Kaggle, extract its CSV file(s) into a folder named `apple_tweets_data` beside this notebook. Otherwise, the code downloads it using KaggleHub.

In [ ]:
dataset_id = "seriousran/appletwittersentimenttexts"
local_folder = Path("apple_tweets_data")

if local_folder.is_dir() and any(local_folder.rglob("*.csv")):
    dataset_folder = local_folder
    print("Using the locally downloaded dataset.")
else:
    dataset_folder = Path(kagglehub.dataset_download(dataset_id))

print("Dataset folder:", dataset_folder)

### Locate and Open the CSV
The code lists CSV files before loading the dataset. If the download contains more than one CSV, set `csv_filename` to the intended file's relative path instead of selecting one arbitrarily.

In [ ]:
csv_files = sorted(dataset_folder.rglob("*.csv"))
print("Available CSV files:")
for file in csv_files:
    print("-", file.relative_to(dataset_folder))

csv_filename = None  # Set a relative filename here only if there are multiple CSVs.

if csv_filename is not None:
    csv_path = dataset_folder / csv_filename
elif len(csv_files) == 1:
    csv_path = csv_files[0]
else:
    raise ValueError("Expected one CSV file. Check the list and set csv_filename.")

try:
    df = pd.read_csv(csv_path, encoding="utf-8")
except UnicodeDecodeError:
    # Older Twitter CSV exports may use Windows-1252 rather than UTF-8.
    df = pd.read_csv(csv_path, encoding="cp1252")

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
df.head()

### Select the Tweet Text Column
The code looks for a column named `text`, `tweet`, `tweets`, `tweet_text`, or `content`. If none is found, inspect the displayed columns and set `text_column` to the actual tweet text column.

In [ ]:
print("Columns:", df.columns.tolist())
text_column = None  # Set the exact column name here if automatic selection is ambiguous.

if text_column is None:
    text_columns = [
        column for column in df.columns
        if column.strip().lower() in {"text", "tweet", "tweets", "tweet_text", "content"}
    ]
    if len(text_columns) != 1:
        raise ValueError("Set text_column to the column containing the tweet text.")
    text_column = text_columns[0]

if text_column not in df.columns:
    raise ValueError(f"Column not found: {text_column}")

print("Selected text column:", text_column)
df[[text_column]].head()

### Inspect Missing Text
Missing text is replaced with an empty string for hashtag extraction. All rows, including duplicates, remain in the dataset because the exercise counts occurrences in the provided data.

In [ ]:
missing_text_count = df[text_column].isna().sum()
print(f"Rows with missing text: {missing_text_count:,}")

tweets = df[text_column].fillna("").astype(str)
print(f"Rows analysed: {len(tweets):,}")

## Task 2: Extract and Count Hashtags

### Define the Regular Expression
The pattern `(?<![\w#])#(\w+)` matches a `#` followed by one or more word characters.

- `(?<![\w#])` prevents a match immediately after a word character or another `#`.
- `#` matches the hashtag symbol.
- `(\w+)` captures the hashtag body: letters (including Unicode letters), digits, or underscores.

Punctuation ends a hashtag. A bare `#` is ignored. This is a simple lab extraction rule, not a full implementation of Twitter's hashtag validation. Extracting before punctuation removal preserves the hashtag markers.

In [ ]:
hashtag_pattern = re.compile(r"(?<![\w#])#(\w+)")

example_tweet = "I like #Apple and #iPhone! #APPLE is popular."
example_hashtags = hashtag_pattern.findall(example_tweet)
print("Example tweet:", example_tweet)
print("Extracted hashtags:", example_hashtags)

### Extract and Normalize Hashtags
Lowercasing combines case variants such as `#Apple` and `#APPLE`. Repeated hashtags in a single tweet count as separate occurrences.

In [ ]:
hashtags_per_tweet = tweets.apply(
    lambda tweet: [tag.lower() for tag in hashtag_pattern.findall(tweet)]
)

all_hashtags = [
    hashtag
    for tweet_hashtags in hashtags_per_tweet
    for hashtag in tweet_hashtags
]
hashtag_counts = Counter(all_hashtags)

print("First 10 extracted hashtags:", [f"#{tag}" for tag in all_hashtags[:10]])

### Report the Hashtag Counts
Total occurrences, unique hashtags, and tweets containing hashtags answer different questions, so they are reported separately.

In [ ]:
total_hashtags = len(all_hashtags)
unique_hashtags = len(hashtag_counts)
tweets_with_hashtags = sum(bool(tags) for tags in hashtags_per_tweet)

print(f"Total hashtag occurrences: {total_hashtags:,}")
print(f"Unique hashtags (ignoring case): {unique_hashtags:,}")
print(f"Tweets containing at least one hashtag: {tweets_with_hashtags:,}")

### Display the Top 10 Hashtags
Sort by frequency from highest to lowest. Alphabetical order breaks ties so the output is reproducible. If fewer than 10 unique hashtags exist, the table displays all of them.

In [ ]:
ranked_hashtags = sorted(
    hashtag_counts.items(),
    key=lambda item: (-item[1], item[0]),
)

top_10_hashtags = pd.DataFrame(
    [(f"#{tag}", count) for tag, count in ranked_hashtags[:10]],
    columns=["Hashtag", "Occurrences"],
)
top_10_hashtags.index = pd.RangeIndex(1, len(top_10_hashtags) + 1, name="Rank")
top_10_hashtags

### Summarize the Results
This cell builds the summary from the actual computed results rather than assuming which hashtag is most frequent.

In [ ]:
if ranked_hashtags:
    highest_count = ranked_hashtags[0][1]
    most_frequent = [
        f"#{tag}" for tag, count in ranked_hashtags if count == highest_count
    ]
    print(f"Found {total_hashtags:,} hashtag occurrences across {tweets_with_hashtags:,} tweets.")
    print(f"There are {unique_hashtags:,} unique hashtags after lowercasing.")
    print("Most frequent hashtag(s):", ", ".join(most_frequent))
    print(f"Occurrences of each most frequent hashtag: {highest_count:,}")
else:
    print("No hashtags matched the extraction rule in the selected text column.")

## Conclusion
This workflow uses regular expressions to extract hashtags, lowercasing to combine case variants, and frequency counting to identify the top 10. Tokenization, stop-word removal, stemming, and lemmatization are discussed in the lab lesson, but are unnecessary for this hashtag exercise and could change the hashtag names. Run every cell with the actual dataset and save the displayed results before submitting.